# Module 05 — Robustness, shift, and honest reporting

> **Aquatic Guardian · Learn computer vision** — Module 05 of 09.  
> **Time:** about 2 hours. **Runs on:** free Google Colab, CPU is enough. **Skip if:** you routinely report metrics by condition and by site, and write data and model cards.

**How to use this notebook.** In Colab choose **File → Save a copy in Drive** first, so your work is kept.
Then **Runtime → Run all**. Every cell runs without errors before you touch anything; the exercises print
feedback rather than crashing. Cells are marked:

- ▶ **Run and read** — nothing to change.
- ✏️ **Predict, then change** — write down what you expect, change one value, re-run.
- 🧩 **Your turn** — replace `None` with your code. A checker tells you whether it works. Hidden solutions follow.

### By the end you can
1. Report performance **by condition** (clear, turbid, low light, blur, far away), not as one number.
2. Measure **distribution shift**: train in one habitat, test in another.
3. Use **augmentation** and say honestly what it fixes and what it doesn't.
4. Build an **image-quality gate** that routes unusable frames to `unknown` instead of guessing.
5. Write an error taxonomy, a data card and a model card, and emit a conservative **event record**.

This module covers **Mentee 1, Tasks 1, 8, 9 and 10**, and the event schema in the **Weeks 1–4 scaffold**.

In [ ]:
#@title ▶ Setup: scenes and real wildlife photographs (run it; you don't need to read it yet)
import numpy as np, matplotlib.pyplot as plt, urllib.request
from pathlib import Path
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10})

PHOTO_BASE = "https://raw.githubusercontent.com/ejri/aquatic-guardian-learning-modules/main/images/"
PHOTO_FILES = {
    "reef-school": "reef-school.jpg", "manatees": "manatees.jpg", "sea-lion": "sea-lion.jpg",
    "nurse-shark": "nurse-shark.jpg", "eagle-ray": "eagle-ray.jpg", "grouper": "grouper.jpg",
    "angelfish": "angelfish.jpg", "turtle": "turtle.jpg", "noaa-honu": "noaa-honu.jpg",
    "noaa-whale": "noaa-whale.jpg", "kelp-forest": "kelp-forest.jpg", "zebra-shark": "zebra-shark.jpg",
}

def load_photo(name, max_side=480):
    """Load a real wildlife JPEG. Looks in ./images and ../images, then downloads from GitHub."""
    file = PHOTO_FILES.get(name, name if str(name).endswith(".jpg") else str(name) + ".jpg")
    arr = None
    for candidate in (Path("images") / file, Path("../images") / file, Path(file)):
        if candidate.is_file():
            arr = plt.imread(candidate)
            break
    if arr is None:
        urllib.request.urlretrieve(PHOTO_BASE + file, file)
        arr = plt.imread(file)
    if getattr(arr, "dtype", None) == np.uint8:
        arr = arr.astype(float) / 255.0
    arr = np.asarray(arr[..., :3], dtype=float)
    h, w = arr.shape[:2]
    scale = max_side / max(h, w)
    if scale < 1:
        yy = (np.linspace(0, h - 1, max(1, int(h * scale)))).astype(int)
        xx = (np.linspace(0, w - 1, max(1, int(w * scale)))).astype(int)
        arr = arr[yy][:, xx]
    return arr


# Illustrative attenuation coefficients beta (1/m) for R, G, B, and the colour of the veiling light.
# These are round teaching numbers with the right ordering, not measurements of any real site.
WATER = {
    "clear_ocean":    dict(beta=(0.40, 0.07, 0.04), veil=(0.02, 0.20, 0.42)),
    "coastal":        dict(beta=(0.55, 0.20, 0.30), veil=(0.05, 0.32, 0.27)),
    "turbid_harbour": dict(beta=(0.95, 0.60, 0.75), veil=(0.14, 0.30, 0.22)),
}

def _fish_mask(H, W, cx, cy, L, ang, aspect=4.0):
    """Ellipse body plus a triangular tail. L is total length (snout to tail tip) in pixels."""
    yy, xx = np.mgrid[0:H, 0:W]
    a = L / 2.25; b = a / aspect
    X = (xx - cx) * np.cos(ang) + (yy - cy) * np.sin(ang)
    Y = -(xx - cx) * np.sin(ang) + (yy - cy) * np.cos(ang)
    body = (X / a) ** 2 + (Y / b) ** 2 <= 1
    tx = -X - 0.80 * a
    tail = (tx >= 0) & (tx <= 0.45 * a) & (np.abs(Y) <= 1.3 * b * tx / (0.45 * a))
    return body | tail, X, Y, a, b

FISH_COLOURS = [(0.97, 0.55, 0.12), (0.96, 0.84, 0.22), (0.80, 0.82, 0.84), (0.93, 0.35, 0.25)]

def make_scene(n_fish=4, H=120, W=180, seed=0, bg_seed=None, fish_dist=(2.0, 6.0), bg_dist=12.0,
               length_px=(22, 40), min_gap=0.55, touching=False, aspect=(4.0, 4.0)):
    """Returns a dict with the true scene radiance J (what you'd see in air), a distance map in metres,
    one boolean mask and one box (x1, y1, x2, y2) per fish, and per-fish truth."""
    r = np.random.default_rng(seed)
    rb = np.random.default_rng(seed if bg_seed is None else bg_seed)
    yy, xx = np.mgrid[0:H, 0:W] / max(H, W)
    J = np.stack([0.48 + 0.10 * yy, 0.43 + 0.08 * yy, 0.34 + 0.04 * xx], -1)       # reef wall / sand
    J = J + 0.04 * rb.standard_normal((H, W, 1)) + 0.015 * rb.standard_normal((H, W, 3))
    dist = np.full((H, W), float(bg_dist))
    masks, boxes, truth, placed = [], [], [], []
    for i in range(n_fish):
        for _ in range(200):
            L = r.uniform(*length_px); ang = r.uniform(-0.6, 0.6) + (np.pi if r.random() < 0.5 else 0)
            cx = r.uniform(0.15 * W, 0.85 * W); cy = r.uniform(0.18 * H, 0.82 * H)
            gap = 0.25 if touching else min_gap
            if all((cx - px) ** 2 + (cy - py) ** 2 > (gap * (L + pl)) ** 2 for px, py, pl in placed):
                break
        placed.append((cx, cy, L))
        asp = aspect[0] if aspect[0] == aspect[1] else r.uniform(*aspect)
        m, X, Y, a, b = _fish_mask(H, W, cx, cy, L, ang, asp)
        col = np.array(FISH_COLOURS[i % len(FISH_COLOURS)])
        J[m] = col
        stripe = m & (np.abs(X - 0.15 * a) < 0.09 * a)
        J[stripe] = (0.95, 0.95, 0.92)
        eye = (X - 0.62 * a) ** 2 + (Y - 0.15 * b) ** 2 < (0.12 * b + 0.6) ** 2
        J[eye & m] = (0.05, 0.05, 0.08)
        d = r.uniform(*fish_dist); dist[m] = d
        ys, xs = np.where(m)
        masks.append(m); boxes.append((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
        truth.append(dict(cx=float(cx), cy=float(cy), length_px=float(L), angle=float(ang), dist_m=float(d), aspect=float(asp)))
    return dict(J=np.clip(J, 0, 1), dist=dist, masks=masks, boxes=boxes, truth=truth)

def underwater(J, dist, water="coastal", light=1.0, photons=None, seed=0):
    """Simple image-formation model, per colour channel c:
         I_c = light * ( J_c * t_c  +  B_c * (1 - t_c) ),   t_c = exp(-beta_c * distance)
    photons: if set, adds shot noise (fewer photons -> noisier). Lower `light` means fewer photons."""
    beta = np.array(WATER[water]["beta"]); B = np.array(WATER[water]["veil"])
    t = np.exp(-beta[None, None, :] * dist[..., None])
    I = light * (J * t + B * (1 - t))
    if photons:
        rng = np.random.default_rng(seed)
        n = photons * light
        I = rng.poisson(np.clip(I, 0, None) / max(light, 1e-6) * n) / n * light
    return np.clip(I, 0, 1)

def degrade(img, turbidity=0.0, low_light=0.0, blur_k=1):
    """The degradation utility used in the Aquatic Guardian project notebooks (copied verbatim in spirit)."""
    out = img.copy()
    if blur_k > 1:
        pad = blur_k // 2; P = np.pad(out, ((pad, pad), (pad, pad), (0, 0)), mode="edge"); acc = np.zeros_like(out)
        for dy in range(blur_k):
            for dx in range(blur_k): acc += P[dy:dy + out.shape[0], dx:dx + out.shape[1], :]
        out = acc / (blur_k * blur_k)
    if turbidity > 0:
        veil = np.array([0.20, 0.42, 0.45]); out = (1 - turbidity) * out + turbidity * veil
        out = (1 - 0.5 * turbidity) * out + 0.5 * turbidity * out.mean()
    if low_light > 0: out = (out * (1 - 0.85 * low_light)) ** (1 + 0.6 * low_light)
    return np.clip(out, 0, 1)

def show(images, titles=None, size=3.2, cmap=None):
    images = images if isinstance(images, (list, tuple)) else [images]
    fig, ax = plt.subplots(1, len(images), figsize=(size * len(images), size * 0.72))
    ax = np.atleast_1d(ax)
    for i, (a, im) in enumerate(zip(ax, images)):
        a.imshow(im, cmap=cmap if im.ndim == 2 else None, vmin=0 if im.ndim == 2 and cmap else None,
                 vmax=1 if im.ndim == 2 and cmap else None)
        a.set_title(titles[i] if titles else ""); a.axis("off")
    plt.tight_layout(); plt.show()

def draw_boxes(img, boxes, colour=(1.0, 0.85, 0.0)):
    out = img.copy()
    for (x1, y1, x2, y2) in boxes:
        x2 -= 1; y2 -= 1
        out[y1, x1:x2 + 1] = colour; out[y2, x1:x2 + 1] = colour
        out[y1:y2 + 1, x1] = colour; out[y1:y2 + 1, x2] = colour
    return out
print("Setup ready. Water types:", list(WATER))

In [ ]:
#@title ▶ Setup: a staged dataset of short camera clips (fish / non_fish / empty_frame)
CLASSES = ["fish", "non_fish", "empty_frame"]     # Mentee 1's Level-1 classes
HW = 48                                           # frame size in pixels
HABITATS = {   # background colour in air, water type, typical object distance (m)
    "reef":    dict(bg=(0.52, 0.47, 0.36), water="clear_ocean",    dist=(1.0, 4.0)),
    "kelp":    dict(bg=(0.30, 0.34, 0.22), water="coastal",        dist=(1.0, 3.5)),
    "harbour": dict(bg=(0.36, 0.33, 0.28), water="turbid_harbour", dist=(0.6, 2.2)),
}

def new_session(cls, habitat, r):
    h = HABITATS[habitat]
    return dict(cls=cls, habitat=habitat, water=h["water"], bg=np.array(h["bg"]) + r.normal(0, 0.03, 3),
                obj_dist=r.uniform(*h["dist"]), bg_dist=r.uniform(6, 12), photons=r.uniform(250, 900),
                x0=r.uniform(14, 34), y0=r.uniform(14, 34), vx=r.normal(0, 0.6), vy=r.normal(0, 0.3),
                L=r.uniform(16, 30), ang=r.uniform(0, np.pi), aspect=r.uniform(3.2, 4.5),
                col=np.array(FISH_COLOURS[r.integers(len(FISH_COLOURS))]) + r.normal(0, 0.05, 3),
                blob_col=r.uniform(0.25, 0.6, 3), blob_r=r.uniform(4, 8), blob_aspect=r.uniform(1.0, 1.5))

def render_frame(s, i, r):
    yy, xx = np.mgrid[0:HW, 0:HW] / HW
    J = s["bg"] + 0.06 * np.stack([yy, 0.8 * yy, 0.5 * xx], -1) + 0.03 * r.standard_normal((HW, HW, 1))
    dist = np.full((HW, HW), s["bg_dist"])
    cx = np.clip(s["x0"] + s["vx"] * i, 8, HW - 8); cy = np.clip(s["y0"] + s["vy"] * i, 8, HW - 8)
    if s["cls"] == "fish":
        m = _fish_mask(HW, HW, cx, cy, s["L"], s["ang"] + 0.02 * i, s["aspect"])[0]; J[m] = s["col"]; dist[m] = s["obj_dist"]
    elif s["cls"] == "non_fish":     # crab, rock, jellyfish: compact blobs
        m = _fish_mask(HW, HW, cx, cy, 2.25 * s["blob_r"], s["ang"], s["blob_aspect"])[0]
        m &= ((np.mgrid[0:HW, 0:HW][1] - cx) ** 2 + (np.mgrid[0:HW, 0:HW][0] - cy) ** 2) <= (1.2 * s["blob_r"]) ** 2
        J[m] = s["blob_col"]; dist[m] = s["obj_dist"]
    return underwater(np.clip(J, 0, 1), dist, s["water"], photons=s["photons"], seed=int(r.integers(1 << 30)))

def make_clips(n_sessions_per_class=40, frames_per_clip=12, habitats=("reef", "kelp", "harbour"), seed=0):
    """Each session is one short clip: one object (or none) drifting past a fixed camera.
    Returns images X (N, HW, HW, 3), labels y (N,), session ids (N,), habitat names (N,)."""
    r = np.random.default_rng(seed); X, y, sess, hab = [], [], [], []
    sid = 0
    for ci, cls in enumerate(CLASSES):
        for k in range(n_sessions_per_class):
            habitat = habitats[k % len(habitats)]
            s = new_session(cls, habitat, r)
            for i in range(frames_per_clip):
                X.append(render_frame(s, i, r)); y.append(ci); sess.append(sid); hab.append(habitat)
            sid += 1
    return np.array(X), np.array(y), np.array(sess), np.array(hab)
print("Clip generator ready. Classes:", CLASSES)

In [ ]:
import pandas as pd, json
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def features(img, fg_thresh=0.10):            # same features as Module 04
    corners = np.concatenate([img[:6, :6].reshape(-1, 3), img[:6, -6:].reshape(-1, 3), img[-6:, :6].reshape(-1, 3), img[-6:, -6:].reshape(-1, 3)])
    bgc = np.median(corners, 0); mask = np.sqrt(((img - bgc) ** 2).sum(-1)) > fg_thresh; area = mask.mean()
    if mask.sum() >= 5:
        ys, xs = np.where(mask); ev = np.clip(np.linalg.eigvalsh(np.cov(np.stack([xs - xs.mean(), ys - ys.mean()]))), 1e-6, None)
        elong = np.sqrt(ev[1] / ev[0]); mr, mg, mb = img[mask].mean(0) - bgc
    else: elong = mr = mg = mb = 0.0
    g = img.mean(-1); edge = np.abs(np.diff(g, axis=1)).mean() + np.abs(np.diff(g, axis=0)).mean()
    return np.array([area, elong, mr, mg, mb, edge])
feat = lambda imgs: np.array([features(im) for im in imgs])

Xtr, ytr, str_, htr = make_clips(n_sessions_per_class=36, seed=10)      # training sessions
Xte, yte, ste, hte = make_clips(n_sessions_per_class=15, seed=99)        # NEW sessions for testing
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(feat(Xtr), ytr)
print(f"clean test accuracy on new sessions: {model.score(feat(Xte), yte):.3f}")

## 1. One number hides everything: evaluate by condition ▶

We apply the project's `degrade()` utility to the test frames, and also generate genuinely **farther** objects with
the physics model from Module 02. Same model, same test sessions, different conditions.

In [ ]:
HABITATS["reef_far"] = dict(HABITATS["reef"], dist=(5.0, 8.0))
HABITATS["harbour_far"] = dict(HABITATS["harbour"], dist=(2.5, 4.0))
Xfar, yfar, _, _ = make_clips(n_sessions_per_class=15, habitats=("reef_far", "harbour_far"), seed=7)

CONDITIONS = {
    "clean": dict(), "turbid (degrade 0.3)": dict(turbidity=0.3), "low light (degrade 0.5)": dict(low_light=0.5),
    "blur (k=3)": dict(blur_k=3), "turbid + blur": dict(turbidity=0.3, blur_k=3),
}
def by_condition(m):
    rows = [dict(condition=k, accuracy=m.score(feat([degrade(im, **kw) for im in Xte]), yte)) for k, kw in CONDITIONS.items()]
    rows.append(dict(condition="farther objects (physics)", accuracy=m.score(feat(Xfar), yfar)))
    return pd.DataFrame(rows).set_index("condition")["accuracy"]
rob = by_condition(model)
ax = rob.plot.barh(figsize=(6, 2.8), color="#2f6f8f"); ax.set_xlim(0, 1); ax.set_xlabel("accuracy"); ax.invert_yaxis()
ax.set_title("Same model, six conditions"); plt.tight_layout(); plt.show()
rob.round(3)

✏️ **Interpret:** which condition hurts most? Look back at the `features()` function and explain the mechanism in
one sentence (what does turbidity do to the foreground mask?).

## 2. Distribution shift: a new habitat ▶

Train only on clear **reef** clips, then test on **harbour** clips. This is what happens when a model built on a
public dataset from one coast is pointed at a different site.

In [ ]:
def split(X, y, h, habs):
    sel = np.isin(h, habs); return X[sel], y[sel]
rows = []
for train_habs in [("reef",), ("reef", "kelp"), ("reef", "kelp", "harbour")]:
    Xa, ya = split(Xtr, ytr, htr, train_habs)
    m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(feat(Xa), ya)
    for test_hab in ("reef", "kelp", "harbour"):
        Xb, yb = split(Xte, yte, hte, (test_hab,))
        rows.append(dict(trained_on="+".join(train_habs), tested_on=test_hab, accuracy=m.score(feat(Xb), yb)))
shift = pd.DataFrame(rows).pivot(index="trained_on", columns="tested_on", values="accuracy").round(3)
shift

Read the table across each row. The gap between "tested on what you trained on" and "tested somewhere new" is
the number you must report, because the project's cameras will not sit on the reef your dataset came from.

## 3. Augmentation ▶

Add degraded copies of training frames so the model sees turbid and dark examples during training.

In [ ]:
r = np.random.default_rng(3); aug_X, aug_y = list(Xtr), list(ytr)
for im, c in zip(Xtr, ytr):
    kw = [dict(turbidity=0.35), dict(low_light=0.5), dict(turbidity=0.25, blur_k=3), dict(blur_k=3)][r.integers(4)]
    aug_X.append(degrade(im, **kw)); aug_y.append(c)
model_aug = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(feat(aug_X), np.array(aug_y))
comparison = pd.DataFrame({"baseline": rob, "augmented": by_condition(model_aug)}).round(3)
comparison

Augmentation usually recovers some degraded-condition accuracy, and often costs some clean accuracy: a 6-feature
linear model has little room to learn both. Report both columns, not just the one that improved.

Now look at the last row. Augmenting with `degrade()` barely helps with objects that are physically farther away.
Training on one degradation model doesn't buy robustness to a different one, which is why Module 02 asked you to
know what `degrade()` assumes.

## 4. An image-quality gate ▶

The scaffold's event schema labels **image quality** (`good | degraded | unusable`) separately from the organism,
so a bad frame is never silently treated as a confident detection. We score quality from three cheap statistics
and set limits from the **training** frames.

In [ ]:
from scipy import ndimage
RING = np.zeros((HW, HW), bool); RING[:6] = RING[-6:] = True; RING[:, :6] = RING[:, -6:] = True
def quality_stats(img):
    """Measured on the 6-pixel border ring, where there is usually only background. That keeps the gate about
    the IMAGE, not about what's in it (otherwise empty frames would look 'low contrast' and get flagged)."""
    g = img.mean(-1); lap = ndimage.laplace(ndimage.gaussian_filter(g, 0.5))
    return dict(brightness=g[RING].mean(), texture=g[RING].std(), sharpness=lap[RING].var())
qs_train = pd.DataFrame([quality_stats(im) for im in Xtr])
LIMITS = qs_train.quantile(0.02)          # anything below the 2nd percentile of training frames is out of range
print("limits (2nd percentile of training frames):"); print(LIMITS.round(5).to_string())

def quality(img):
    q = quality_stats(img); low = [k for k in LIMITS.index if q[k] < LIMITS[k]]
    return "unusable" if len(low) >= 2 else ("degraded" if len(low) == 1 else "good")

rows = []
for k, kw in CONDITIONS.items():
    imgs = [degrade(im, **kw) for im in Xte]; q = np.array([quality(im) for im in imgs])
    pred = model.predict(feat(imgs)); ok = q == "good"
    rows.append(dict(condition=k, share_good=ok.mean(), acc_on_good=(pred[ok] == yte[ok]).mean() if ok.any() else np.nan,
                     acc_on_flagged=(pred[~ok] == yte[~ok]).mean() if (~ok).any() else np.nan))
pd.DataFrame(rows).set_index("condition").round(3)

✏️ **Interpret:** three things to look for.

- Where some frames pass and some are flagged, is accuracy lower on the flagged ones? Then the gate is routing the
  riskiest frames to `unknown`/human review, which is its job.
- Where the gate flags *every* frame (`NaN` under `acc_on_good`), the whole condition goes to review. Safe, but costly.
- In clean water the gate still flags a few frames that the model would have got right. That's its false-alarm cost.
  Write all three into your report.

## 5. Error taxonomy ▶

Bucket every mistake by (true class, predicted class, condition). Patterns here tell you what data to collect next.

In [ ]:
errs = []
for k, kw in CONDITIONS.items():
    imgs = [degrade(im, **kw) for im in Xte]; pred = model.predict(feat(imgs))
    for t, p in zip(yte, pred):
        if t != p: errs.append(dict(true=CLASSES[t], predicted=CLASSES[p], condition=k))
taxonomy = pd.DataFrame(errs).value_counts().rename("count").reset_index().head(10)
taxonomy

## 6. Data card, model card, and a conservative event record ▶

These go in the repo with the code. The event record mirrors the Weeks 1–4 scaffold: when evidence is weak or the
frame is degraded, the defaults are `unknown` and `human_review_required`, **never** a "safe" state.

In [ ]:
data_card = {
    "name": "Synthetic staged clips (teaching)", "source": "generated in this notebook", "licence": "n/a (synthetic)",
    "classes": CLASSES, "split": "by session (clip), never by frame",
    "habitats": sorted(set(htr)), "known_gaps": ["no real organisms", "no occlusion between animals", "fixed camera"],
}
model_card = {
    "model": "standardised 6-feature logistic regression", "trained_on": data_card["name"],
    "metrics_by_condition": rob.round(3).to_dict(), "metrics_by_habitat_shift": shift.to_dict(),
    "intended_use": "teaching; screening frames for human review",
    "must_not_be_used_for": None,            # 🧩 fill this in (a list of strings)
}
def event_record(img, model):
    q = quality(img)
    proba = model.predict_proba(feat([img]))[0]; top = int(proba.argmax())
    confident = (q == "good") and proba[top] >= 0.8
    return {
        "image_quality": q, "detections": [dict(cls=CLASSES[top], confidence=round(float(proba[top]), 3))],
        "event_type": "normal_activity" if confident else "unknown",
        "action": "log" if confident else "human_review_required",
    }
print(json.dumps(event_record(Xte[0], model), indent=1))
print(json.dumps(event_record(degrade(Xte[0], turbidity=0.7, blur_k=7), model), indent=1))

### 🧩 Exercise — the most important field

Fill in `model_card["must_not_be_used_for"]` with at least two specific uses this model must not support. One should
concern ecological or regulatory claims.

In [ ]:
def check_card(card):
    v = card.get("must_not_be_used_for")
    if not v: print("Not yet: add a list of at least two strings."); return
    text = " ".join(v).lower()
    if len(v) >= 2 and any(w in text for w in ("ecolog", "safe", "permit", "regulat", "impact", "health", "compliance")):
        print("✅ Good. Keep this list in the repo next to the metrics.")
    else: print("Add at least two items, including one about ecological, safety or regulatory claims.")
check_card(model_card)

In [ ]:
#@title Example answer (double-click to show)
model_card["must_not_be_used_for"] = [
    "claiming an ecosystem is healthy or unharmed (absence of detections is not absence of animals)",
    "permit or compliance decisions without detection-probability correction and human review",
    "species identification (classes are fish / non_fish / empty only)",
    "sites or water conditions outside the evaluated habitats without re-validation",
]
check_card(model_card)
with open("model_card.json", "w") as f: json.dump(model_card, f, indent=1, default=str)
print("saved model_card.json")

## Check yourself

<details><summary>1. Your model has 0.95 accuracy overall and 0.55 in turbid water. What's the headline?</summary>Both numbers, with the condition named. The overall figure alone would mislead anyone deploying the model where turbidity is common.</details>
<details><summary>2. Why set quality limits from training frames rather than test frames?</summary>The gate must be fixed before seeing test data, otherwise the evaluation is tuned to the test set.</details>
<details><summary>3. What should an event record say when the frame is degraded and confidence is 0.62?</summary>event_type "unknown", action "human_review_required". Never a safe or normal state by default.</details>
<details><summary>4. Augmentation raised turbid accuracy from 0.5 to 0.7. Is the turbidity problem solved?</summary>No. It's reduced. Report the remaining gap and the clean-accuracy change, and keep the quality gate.</details>

**Next:** Module 06, *Neural networks and CNNs* — a model that learns its own features, and whether that helps under turbidity.